In [1]:
%%writefile app.py
import streamlit as st
from huggingface_hub import InferenceClient
from PIL import Image, ImageFilter, ImageOps
import io

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxx"

# Models
CHAT_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ART_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

# --- PAGE SETUP ---
st.set_page_config(page_title="Week 11: Style Enhancement", layout="wide")
st.title("🚀 GenAI + Image Processing App")
st.markdown("Week 11: Integrated AI generation with classic image filtering (Pillow).")

# --- AUTHENTICATION ---
if not HF_TOKEN.startswith("hf_"):
    st.error("Error: Please provide a valid Hugging Face Token in the code.")
    st.stop()

client = InferenceClient(token=HF_TOKEN)

# --- HELPER FUNCTION: FILTERS ---
def apply_filter(image, filter_type):
    """
    Applies a specific filter to the PIL image using Pillow library.
    """
    if filter_type == "Grayscale":
        return ImageOps.grayscale(image)
    elif filter_type == "Blur":
        return image.filter(ImageFilter.GaussianBlur(radius=2))
    elif filter_type == "Contour":
        return image.filter(ImageFilter.CONTOUR)
    elif filter_type == "Invert":
        return ImageOps.invert(image)
    elif filter_type == "Sharpen":
        return image.filter(ImageFilter.SHARPEN)
    else:
        return image  # Original

# --- SIDEBAR ---
mode = st.sidebar.radio("Select Mode:", ["💬 Chat Mode", "🎨 Art Mode"])

# --- CHAT MODE ---
if mode == "💬 Chat Mode":
    st.header("💬 Chatbot (Mistral 7B)")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for msg in st.session_state.messages:
        st.chat_message(msg["role"]).markdown(msg["content"])

    if prompt := st.chat_input("Ask something..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        st.chat_message("user").markdown(prompt)

        with st.chat_message("assistant"):
            try:
                response = ""
                for token in client.chat_completion([{"role": "user", "content": prompt}], model=CHAT_MODEL, max_tokens=500, stream=True):
                    if token.choices[0].delta.content:
                        response += token.choices[0].delta.content
                st.markdown(response)
                st.session_state.messages.append({"role": "assistant", "content": response})
            except Exception as e:
                st.error(f"Error: {e}")

# --- ART MODE (UPDATED FOR WEEK 11) ---
elif mode == "🎨 Art Mode":
    st.header("🎨 Art Generator & Style Filters")

    # Session state to hold the generated image
    if "generated_image" not in st.session_state:
        st.session_state.generated_image = None

    prompt = st.text_input("Describe image:", "A futuristic cyberpunk city with neon lights")

    if st.button("Generate New Image"):
        if prompt:
            with st.spinner("Generating image on Cloud..."):
                try:
                    image = client.text_to_image(prompt, model=ART_MODEL)
                    st.session_state.generated_image = image # Save to session
                except Exception as e:
                    st.error(f"Error: {e}")

    # Display and Filter Section
    if st.session_state.generated_image is not None:
        st.subheader("Image Editor")

        col1, col2 = st.columns([1, 1])

        with col1:
            st.image(st.session_state.generated_image, caption="Original Output", use_container_width=True)

        with col2:
            # Filter Selection
            filter_choice = st.selectbox(
                "Select a Filter:",
                ["Original", "Grayscale", "Blur", "Contour", "Invert", "Sharpen"]
            )

            # Apply Filter
            filtered_image = apply_filter(st.session_state.generated_image, filter_choice)
            st.image(filtered_image, caption=f"Filtered: {filter_choice}", use_container_width=True)

            # Download Button
            buf = io.BytesIO()
            filtered_image.save(buf, format="PNG")
            st.download_button("Download Filtered Image", data=buf.getvalue(), file_name="filtered_art.png", mime="image/png")

Writing app.py


In [2]:
!pip install huggingface_hub streamlit pillow -q
!npm install -g localtunnel
!wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb > cloudflared.deb
!dpkg -i cloudflared.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 152.5 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋npm notice
npm notice New major version of npm available! 10.8.2 -> 11.7.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.7.0
npm notice To update run: npm install -g npm@11.7.0
npm notice
⠙Selecting previously unselected package cloudflared.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2025.11.1) ...
Setting up cloudflared (2025.11.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!streamlit run app.py & cloudflared tunnel --url http://localhost:8501

2026-01-12T14:26:56Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-12T14:26:56Z INF Requesting new quick Tunnel on trycloudflare.com...



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.248.243:8501

2026-01-12T14:26:59Z INF +--------------------------------------------------------------------------------------------+
2026-01-12T14:26:59Z INF